In [ ]:

!pip install -q transformers trl peft accelerate bitsandbytes datasets pyarrow tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 465.5/465.5 kB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 12.7 MB/s eta 0:00:00


In [2]:
!pip install evaluate
!pip install rouge_score
!pip install bert_score
!pip install bitsandBytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=7992d7a3fa7f682f34d457f112a8b9b6796f1ef236205f74746a7d7c7746780e
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 4.8 MB/s eta 0:00:00


In [3]:
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    DataCollatorForLanguageModeling,
    TrainingArguments,
    Trainer
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training
)

In [4]:
import os
if "COLAB_GPU" in os.environ:
  from google.colab import output
  output.enable_custom_widget_manager()

In [ ]:
import os
from huggingface_hub import notebook_login

# If running in Google Colab
if "COLAB_GPU" in os.environ:
    !huggingface-cli login
# If running locally (Jupyter, VS Code, etc.)
else:
    notebook_login() 


⚠️  Warning: 'huggingface-cli login' is deprecated. Use 'hf auth login' instead.

    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? (Y/n) Y
Token is valid (permission: read).
The token `lk` has been saved to /root/.cache/huggingface/stored_tokens
Cannot authenticate through git-credential as no 

In [6]:
from datasets import load_dataset

In [ ]:
dataset = load_dataset("Amod/mental_health_counseling_conversations")
dataset = dataset["train"].train_test_split(test_size=0.30, seed=42)
train_dataset = dataset["train"]
test_dataset = dataset["test"]


# Convert to pandas
train_df = train_dataset.to_pandas()
test_df  = test_dataset.to_pandas()

# Reset index
train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

In [9]:
def format_prompt(example):
    user = example["Context"]
    assistant = example["Response"]

    text = (
        "<|im_start|>user\n"
        + user
        + "<|im_end|>\n"
        + "<|im_start|>assistant\n"
        + assistant
        + "<|im_end|>\n"
    )
    return {"text": text}




In [10]:
train_dataset = train_dataset.map(format_prompt)
test_dataset  = test_dataset.map(format_prompt)


Map:   0%|          | 0/2458 [00:00<?, ? examples/s]

Map:   0%|          | 0/1054 [00:00<?, ? examples/s]

In [ ]:

# TOKENIZER


model_name = "Qwen/Qwen2-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True
)


if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

max_length = 512   # SAME


def tokenize(batch):
    # Tokenize Qwen text 
    enc = tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=max_length,

    )

    enc["labels"] = enc["input_ids"].copy()
    return enc

train_tokenized = train_dataset.map(
    tokenize,
    batched=True,
    remove_columns=train_dataset.column_names
)

test_tokenized = test_dataset.map(
    tokenize,
    batched=True,
    remove_columns=test_dataset.column_names
)

train_tokenized.set_format("torch")
test_tokenized.set_format("torch")


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/2458 [00:00<?, ? examples/s]

Map:   0%|          | 0/1054 [00:00<?, ? examples/s]

In [ ]:

#  QLoRA 4-BIT QUANTIZATION  (Qwen Version)

from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,    
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)


In [ ]:

# LOAD QWEN MODEL IN 4-BIT (QLoRA)

from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model_name = "Qwen/Qwen2-0.5B-Instruct"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
)


model.config.use_cache = False

model = prepare_model_for_kbit_training(model)


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [ ]:

# LoRA CONFIGURATION (Qwen2 Version)

from peft import LoraConfig, get_peft_model

lora_cfg = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM",

    # Qwen2  LoRA target layers
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
    ],
)

# Apply LoRA to Qwen model
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()


trainable params: 2,162,688 || all params: 496,195,456 || trainable%: 0.4359


In [ ]:

# TRAINING ARGUMENTS 

output_dir = "./qlora-qwen2-mental-health"

training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,     
    warmup_steps=50,
    logging_steps=50,
    eval_steps=300,
    eval_strategy="epoch",
    save_strategy="epoch",
    bf16=True,             
    fp16=False,             
    report_to="none",      
    save_total_limit=2,     
)

training_args.label_names = ["labels"]


In [ ]:

# DATA COLLATOR + TRAINER 

from transformers import DataCollatorForLanguageModeling, Trainer

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=test_tokenized,
    data_collator=data_collator,
)


# 10. TRAIN

trainer.train()


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Epoch,Training Loss,Validation Loss
1,2.607600,2.593933
2,2.507200,2.525399
3,2.435900,2.504055


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


TrainOutput(global_step=462, training_loss=2.5602829549219703, metrics={'train_runtime': 6294.3575, 'train_samples_per_second': 1.172, 'train_steps_per_second': 0.073, 'total_flos': 8156431378022400.0, 'train_loss': 2.5602829549219703, 'epoch': 3.0})

In [17]:
# Save LoRA adapter + tokenizer
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)

print("✅ QLoRA fine-tuning finished. Adapter saved to:", output_dir)

✅ QLoRA fine-tuning finished. Adapter saved to: ./qlora-qwen2-mental-health


In [18]:
from huggingface_hub import login
login()

In [ ]:
# #upload login token
# import os
# from huggingface_hub import notebook_login

# # If running in Google Colab
# if "COLAB_GPU" in os.environ:
#     !huggingface-cli login
# # If running locally (Jupyter, VS Code, etc.)
# else:
#     notebook_login() 


In [19]:
from huggingface_hub import upload_folder

upload_folder(
    folder_path="./qlora-qwen2-mental-health",  #
    repo_id="ZunairAhmad/qwen2-mental-health",
    repo_type="model"
)

print("✔ Uploaded successfully!")


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          | 21.4kB / 8.68MB            

  ...adapter_model.safetensors:   0%|          | 21.4kB / 8.68MB            

  ...adapter_model.safetensors:   0%|          | 21.4kB / 8.68MB            

...

  ...kpoint-462/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            

  ...tal-health/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            

  ...eckpoint-308/scheduler.pt: 100%|##########| 1.47kB / 1.47kB            

  ...kpoint-308/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            

✔ Uploaded successfully!
